## power demand EDA + prediction

using pandas and numpy mostly, going to merge 3 datasets (demand, weather, economic) and try to predict hourly power demand

In [2]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor

In [3]:
demand_raw = pd.read_csv('PGCB_date_power_demand.csv')
demand_raw = demand_raw[['datetime', 'demand_mw']].copy()
demand_raw['datetime'] = pd.to_datetime(demand_raw['datetime'])

demand_raw.head()

,datetime,demand_mw
0,2015-04-19 22:00:00,6323
1,2015-04-19 21:00:00,6667
2,2015-04-19 19:00:00,6897
3,2015-04-19 18:30:00,6933
4,2015-04-19 18:00:00,6874


In [4]:
demand_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 92650 entries, 0 to 92649
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   datetime   92650 non-null  datetime64[us]
 1   demand_mw  92650 non-null  int64         
dtypes: datetime64[us](1), int64(1)
memory usage: 1.4 MB


loading weather data - the column names in this csv are completely wrong so had to rename manually

In [5]:
climate_raw = pd.read_csv('weather_data.csv')

# the original col names make no sense, fixing them
col_map = {
    'latitude': 'time',
    'utc_offset_seconds': 'apparent_temperature (°C)',
    'timezone': 'precipitation (mm)',
    'Unnamed: 9': 'sunshine_duration (s)',
    'elevation': 'relative_humidity_2m (%)'
}
climate_raw.rename(columns=col_map, inplace=True)

keep_cols = ['time', 'apparent_temperature (°C)', 'precipitation (mm)', 'sunshine_duration (s)', 'relative_humidity_2m (%)']
climate_raw = climate_raw[keep_cols].iloc[3:].reset_index(drop=True)  # first 3 rows are garbage
climate_raw['time'] = pd.to_datetime(climate_raw['time'])

climate_raw.head()

C:\Users\airfo\AppData\Local\Temp\ipykernel_27060\687398855.py:1: DtypeWarning: Columns (0: longitude, 1: elevation, 2: utc_offset_seconds, 3: timezone, 4: timezone_abbreviation, 5: Unnamed: 6, 6: Unnamed: 7, 7: Unnamed: 8, 8: Unnamed: 9) have mixed types. Specify dtype option on import or set low_memory=False.
  climate_raw = pd.read_csv('weather_data.csv')


,time,apparent_temperature (°C),precipitation (mm),sunshine_duration (s),relative_humidity_2m (%)
0,2014-01-01 00:00:00,13.3,0,0,89
1,2014-01-01 01:00:00,13.2,0,0,91
2,2014-01-01 02:00:00,12.8,0,0,91
3,2014-01-01 03:00:00,12.5,0,0,92
4,2014-01-01 04:00:00,12.2,0,0,93


kept temperature, precipitation, humidity and sunshine - these are the ones that should logically affect power consumption (ac usage, heating etc)

In [6]:
econ_raw = pd.read_csv('economic_full_1.csv')

# only keeping 3 indicators, the rest arent really relevant
useful = [
    'Urban population',
    'Rural population',
    'Electric power consumption (kWh per capita)'
]
econ_raw = econ_raw[econ_raw['Indicator Name'].isin(useful)].copy()
econ_raw.head()

,Country Name,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,1966,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
25,X,Urban population,SP.URB.TOTL,2649573.0,2813246.0,2999574.0,3195917.0,3406306.0,3633117.0,3878538.0,...,4.804319e+07,4.898278e+07,4.990327e+07,5.083089e+07,5.174678e+07,5.264356e+07,5.365052e+07,55140091.0,56717264.0,NaN
26,X,Rural population,SP.RUR.TOTL,49179087.0,50497102.0,51881572.0,53308485.0,54772068.0,56269402.0,57791268.0,...,1.127687e+08,1.132034e+08,1.136198e+08,1.140822e+08,1.145512e+08,1.150153e+08,1.157344e+08,116326899.0,116845100.0,NaN
1494,X,Electric power consumption (kWh per capita),EG.USE.ELEC.KH.PC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.127803e+02,4.432129e+02,4.711261e+02,5.151017e+02,5.099459e+02,5.741182e+02,6.026747e+02,NaN,NaN,NaN


from the economic dataset I only took these 3 because they seemed most relevant to predicting demand

In [7]:
# shifting year labels by 1 - economic data for year Y represents year Y+1 basically
src_years = [str(y) for y in range(2014, 2024)]
dst_years = [str(y) for y in range(2015, 2025)]

econ_raw = econ_raw[['Indicator Name'] + src_years].copy()
econ_raw.rename(columns=dict(zip(src_years, dst_years)), inplace=True)
econ_raw.reset_index(drop=True, inplace=True)

econ_raw.head()

,Indicator Name,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Urban population,4.611698e+07,4.708101e+07,4.804319e+07,4.898278e+07,4.990327e+07,5.083089e+07,5.174678e+07,5.264356e+07,5.365052e+07,55140091.0
1,Rural population,1.118479e+08,1.123022e+08,1.127687e+08,1.132034e+08,1.136198e+08,1.140822e+08,1.145512e+08,1.150153e+08,1.157344e+08,116326899.0
2,Electric power consumption (kWh per capita),3.265029e+02,3.489327e+02,4.127803e+02,4.432129e+02,4.711261e+02,5.151017e+02,5.099459e+02,5.741182e+02,6.026747e+02,NaN


economic data is yearly but we need hourly to merge with the other two - so expanding each year's value across all hours of that year and then forward filling

In [8]:
hourly_blocks = []
annual_cols = [str(y) for y in range(2015, 2025)]

for _, record in econ_raw.iterrows():
    ind = record['Indicator Name']
    for yr in annual_cols:
        hrs = pd.date_range(start=f"{yr}-01-01 00:00:00", end=f"{yr}-12-31 23:00:00", freq='h')
        block = pd.DataFrame({'datetime': hrs, 'Indicator Name': ind, 'value': record[yr]})
        hourly_blocks.append(block)

econ_hourly = pd.concat(hourly_blocks, ignore_index=True)
econ_hourly.sort_values(['Indicator Name', 'datetime'], inplace=True)
econ_hourly['value'] = econ_hourly.groupby('Indicator Name')['value'].ffill()

print(econ_hourly.isna().sum())
econ_hourly.head()

datetime          0
Indicator Name    0
value             0
dtype: int64


,datetime,Indicator Name,value
175344,2015-01-01 00:00:00,Electric power consumption (kWh per capita),326.502853
175345,2015-01-01 01:00:00,Electric power consumption (kWh per capita),326.502853
175346,2015-01-01 02:00:00,Electric power consumption (kWh per capita),326.502853
175347,2015-01-01 03:00:00,Electric power consumption (kWh per capita),326.502853
175348,2015-01-01 04:00:00,Electric power consumption (kWh per capita),326.502853


now merging all 3 datasets together and checking correlation with demand_mw

In [9]:
econ_hourly['datetime'] = pd.to_datetime(econ_hourly['datetime'])

merged_tmp = pd.merge(econ_hourly, climate_raw, left_on='datetime', right_on='time', how='inner')

merged_wide = merged_tmp.pivot_table(
    index='datetime', columns='Indicator Name', values='value'
).reset_index()

# reattach weather cols
merged_wide = pd.merge(merged_wide, climate_raw, left_on='datetime', right_on='time', how='inner')
merged_wide.drop(columns=['time'], inplace=True)

combined = pd.merge(merged_wide, demand_raw[['datetime', 'demand_mw']], on='datetime', how='inner')

corr_vals = combined.drop(columns=['datetime']).corr()['demand_mw'].sort_values(ascending=False)
print(corr_vals)
print(combined.shape)

demand_mw                                      1.000000
Rural population                               0.657682
Urban population                               0.657085
Electric power consumption (kWh per capita)    0.645188
apparent_temperature (°C)                      0.465568
sunshine_duration (s)                         -0.003914
precipitation (mm)                            -0.025336
relative_humidity_2m (%)                      -0.052590
Name: demand_mw, dtype: float64
(84448, 9)


rural population, urban population and electric power consumption all have really high corr with demand which makes sense. apparent temp also looks good

now adding lag features so the model can look back in time. using 1h, 24h, 48h lags and 6h/24h rolling averages

In [10]:
combined.sort_values('datetime', inplace=True)
combined.set_index('datetime', inplace=True)

tgt = 'demand_mw'

combined['prev_1h'] = combined[tgt].shift(1)
combined['prev_24h'] = combined[tgt].shift(24)  # same hour yesterday
combined['prev_48h'] = combined[tgt].shift(48)  # 2 days ago

# rolling means - shift by 1 first so we don't leak current value
combined['roll_6h'] = combined[tgt].shift(1).rolling(6).mean()
combined['roll_24h'] = combined[tgt].shift(1).rolling(24).mean()

combined.reset_index(inplace=True)

In [11]:
lag_corr = combined.drop(columns=['datetime']).corr()['demand_mw'].sort_values(ascending=False)
print(lag_corr)

demand_mw                                      1.000000
prev_1h                                        0.901632
roll_6h                                        0.886964
roll_24h                                       0.875019
prev_24h                                       0.866040
prev_48h                                       0.838744
Rural population                               0.657682
Urban population                               0.657085
Electric power consumption (kWh per capita)    0.645188
apparent_temperature (°C)                      0.465568
sunshine_duration (s)                         -0.003914
precipitation (mm)                            -0.025336
relative_humidity_2m (%)                      -0.052590
Name: demand_mw, dtype: float64


corr of all lag features with demand is really high, good sign that the approach is working

dropping the weather features that had low corr - relative humidity, sunshine and precipitation weren't really helping

In [12]:
combined.drop(columns=['relative_humidity_2m (%)', 'sunshine_duration (s)', 'precipitation (mm)', 'datetime'], inplace=True)

# temperature col came in as object for some reason
combined['apparent_temperature (°C)'] = pd.to_numeric(combined['apparent_temperature (°C)'], errors='coerce')

splitting into train/test - using 90/10 split chronologically so 2015-2023 for training and 2024 for testing. can't shuffle because of the lag features

In [13]:
features = combined.drop(columns=['demand_mw'])
labels = combined['demand_mw']

cutoff = int(len(combined) * 0.9)

X_train, X_eval = features.iloc[:cutoff], features.iloc[cutoff:]
y_train, y_eval = labels.iloc[:cutoff], labels.iloc[cutoff:]

print(X_train.shape, X_eval.shape)

(76003, 9) (8445, 9)


using lightgbm - handles non linear stuff well and is pretty fast, should work well here

In [14]:
forecaster = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

forecaster.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [15]:
y_hat = forecaster.predict(X_eval)

In [16]:
mape_score = np.mean(np.abs((y_eval - y_hat) / y_eval)) * 100
print("MAPE:", mape_score)

MAPE: 5.429210782799737


getting around 5% MAPE which is pretty good for this use case

checking which features the model found most useful

In [17]:
feat_imp = pd.Series(forecaster.feature_importances_, index=features.columns)
feat_imp = feat_imp.sort_values(ascending=False)

print(feat_imp.head(15))

prev_1h                                        2843
roll_6h                                        2578
prev_24h                                       2284
apparent_temperature (°C)                      2262
roll_24h                                       2164
prev_48h                                       1833
Electric power consumption (kWh per capita)     567
Rural population                                225
Urban population                                 33
dtype: int32


the rolling averages and prev_1h are the most important features which makes sense - recent demand is the best predictor of next hour demand. population and consumption indicators have decent corr but don't contribute as much to the hourly prediction, which also makes sense since they change very slowly